# Chapter 1 — Responses API Fundamentals · Practice

**Goal:** make real Responses API calls and take each part of the contract apart: request fields, output items, `status` / `incomplete_details`, the three conversation-state patterns, streaming and background mode, token counting and cost, reasoning effort, and the Chat Completions contrast. You finish with a small multi-turn chat helper.

- Theory: [README.md](README.md)
- **Estimated time:** 2–3 hours
- **Estimated API cost:** well under $0.01 with the default model `gpt-6-luna` ($0.10 / $0.50 per million input / output tokens)
- **Prerequisites:** [00 — Prerequisites](../../00-prerequisites/README.md) done, `OPENAI_API_KEY` in `.env`, run from the repo venv

Cells marked **(live)** call the API. Every other cell is offline: it builds payloads or works on mock `Response` objects, so you can still learn the contract without credit.

In [ ]:
import json
import os
import time
from dataclasses import dataclass, field
from types import SimpleNamespace

import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from openai.types.responses import Response

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("Model:", MODEL, "| openai SDK", openai.__version__)

In [ ]:
# Fail fast if the key is missing. Offline cells below still work if you skip this one.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
client = OpenAI()   # reads OPENAI_API_KEY from the environment

### Shared helpers (offline)

`PRICES` comes from the model pages (USD per million tokens, checked 2026-10-03). `mock_response()` builds a real SDK `Response` object from a dict, so offline demos use exactly the same types as live calls.

In [ ]:
PRICES = {  # USD per 1M tokens: input, cached input, cache writes, output
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "cache_write": 12.50, "output": 50.00},
    "gpt-6.1-sol": {"input": 2.00, "cached": 0.10, "cache_write": 2.50, "output": 10.00},
    "gpt-6-sol": {"input": 2.00, "cached": 0.20, "cache_write": 2.50, "output": 10.00},
    "gpt-6-luna": {"input": 0.10, "cached": 0.01, "cache_write": 0.125, "output": 0.50},
}
LONG_CONTEXT_THRESHOLD = 272_000  # GPT-6: above this, 2x input/cache rates and 1.5x output for the whole request


def price_for(model: str) -> dict:
    """Look up a model's prices; dated snapshots (e.g. gpt-6-luna-2026-...) use their base model's row."""
    if model in PRICES:
        return PRICES[model]
    base = max((m for m in PRICES if model.startswith(m + "-")), key=len, default=None)
    if base is None:   # never guess: a silent fallback would under- or over-price unknown models
        raise ValueError(f"No price for model {model!r}; add it to PRICES from its model page")
    return PRICES[base]


def cost_usd(usage, model: str = MODEL) -> float:
    """Estimate the cost of one response from its usage object."""
    p = price_for(model)
    cached = usage.input_tokens_details.cached_tokens
    writes = usage.input_tokens_details.cache_write_tokens
    uncached = usage.input_tokens - cached - writes
    in_mult, out_mult = (2.0, 1.5) if usage.input_tokens > LONG_CONTEXT_THRESHOLD else (1.0, 1.0)
    input_cost = (uncached * p["input"] + cached * p["cached"] + writes * p["cache_write"]) * in_mult
    return (input_cost + usage.output_tokens * p["output"] * out_mult) / 1e6


def show(resp) -> None:
    """Print only what matters about a Response."""
    reason = resp.incomplete_details.reason if resp.incomplete_details else None
    print(f"id={resp.id} status={resp.status} incomplete_reason={reason}")
    print("output item types:", [item.type for item in resp.output])
    print("output_text:", repr(resp.output_text[:300]))
    if resp.usage:
        u = resp.usage
        try:
            cost = f"${cost_usd(u, resp.model):.6f}"
        except ValueError:
            cost = "n/a (model not in PRICES)"
        print(f"usage: in={u.input_tokens} (cached={u.input_tokens_details.cached_tokens}) "
              f"out={u.output_tokens} (reasoning={u.output_tokens_details.reasoning_tokens}) cost≈{cost}")


def mock_response(*, status="completed", reason=None, text=None, refusal=None, function_call=None,
                  reasoning=True, usage=(120, 40, 25), error=None, rid="resp_mock", model="gpt-6-luna"):
    """Build a real SDK Response object without calling the API."""
    output = []
    if reasoning:
        output.append({"id": "rs_1", "type": "reasoning", "summary": []})
    content = []
    if text is not None:
        content.append({"type": "output_text", "text": text, "annotations": []})
    if refusal is not None:
        content.append({"type": "refusal", "refusal": refusal})
    if content:
        output.append({"id": "msg_1", "type": "message", "role": "assistant",
                       "status": "completed" if status == "completed" else "incomplete", "content": content})
    if function_call:
        output.append({"id": "fc_1", "type": "function_call", "call_id": "call_1",
                       "name": function_call, "arguments": "{}", "status": "completed"})
    inp, out, rsn = usage
    return Response.model_validate({
        "id": rid, "object": "response", "created_at": 0, "model": model, "status": status,
        "incomplete_details": {"reason": reason} if reason else None, "error": error,
        "instructions": None, "metadata": {}, "output": output, "parallel_tool_calls": True,
        "temperature": None, "tool_choice": "auto", "tools": [], "top_p": None,
        "usage": {"input_tokens": inp, "input_tokens_details": {"cached_tokens": 0, "cache_write_tokens": 0},
                  "output_tokens": out, "output_tokens_details": {"reasoning_tokens": rsn},
                  "total_tokens": inp + out},
    })


show(mock_response(text="Hello from a mock."))

## 1.1 Anatomy of a Responses API request

One endpoint, `POST /v1/responses`, does everything. You send `model`, `input` (a string or a list of items), optional `instructions`, `reasoning` and `max_output_tokens`. You get back a `Response` whose `output` is a **list of typed items**, plus `status`, `incomplete_details` and `usage`. [Theory](README.md#11-anatomy-of-a-responses-api-request)

In [ ]:
# (live) Your first request
response = client.responses.create(
    model=MODEL,
    instructions="You are a concise assistant for a cloud architecture team.",
    input="In two sentences, what is an API gateway?",
    reasoning={"effort": "low"},
    max_output_tokens=1000,   # includes reasoning tokens
)
show(response)

> **Observe:** the first output item is usually `reasoning`, not `message`. `output_text` joins every `output_text` part for you. `usage.output_tokens` already includes the reasoning tokens.

In [ ]:
# (offline) Why output[0].content[0].text is fragile
r = mock_response(text="An API gateway is a single entry point for clients...")
print("item types:", [item.type for item in r.output])
first = r.output[0]
print("output[0].type:", first.type, "| content:", first.content)   # reasoning item: no text content

# Robust ways to read text
print("output_text:", r.output_text)
texts = [part.text for item in r.output if item.type == "message"
         for part in item.content if part.type == "output_text"]
print("filtered:", texts)

# The same response as JSON: this is what the HTTP body looks like
print(json.dumps(r.model_dump(include={"id", "status", "output"}, exclude_none=True), indent=2)[:700])

In [ ]:
# (live) Errors are HTTP exceptions, not statuses. Catch typed exceptions.
print("SDK default max_retries:", OpenAI(api_key="placeholder").max_retries)
try:
    client.responses.create(model="gpt-does-not-exist", input="hi", max_output_tokens=50)
except openai.NotFoundError as e:
    print("404 NotFoundError:", e.status_code)
except openai.BadRequestError as e:
    print("400 BadRequestError:", e.status_code)
except openai.AuthenticationError as e:
    print("401 AuthenticationError: check OPENAI_API_KEY")
except openai.APIConnectionError:
    print("Network problem: the SDK already retried")

> **Try it:** add `text={"verbosity": "low"}` to the first request and compare `output_tokens`.

## 1.2 Input items, roles and instructions

`input` can be a string (one `user` message) or a list of items. Roles carry authority: `developer`/`system` beat `user`, and `assistant` items are the model's own earlier output. `instructions` is a system/developer message for **this request only**: it is not carried over by `previous_response_id`. [Theory](README.md#12-input-items-roles-and-instructions)

In [ ]:
# (offline) Three ways to say the same thing
a = {"model": MODEL, "instructions": "Talk like a pirate.", "input": "Are semicolons optional in JavaScript?"}
b = {"model": MODEL, "input": [
    {"role": "developer", "content": "Talk like a pirate."},
    {"role": "user", "content": "Are semicolons optional in JavaScript?"},
]}
c = {"model": MODEL, "input": [   # explicit content parts: needed for images and files
    {"role": "developer", "content": [{"type": "input_text", "text": "Talk like a pirate."}]},
    {"role": "user", "content": [{"type": "input_text", "text": "Are semicolons optional in JavaScript?"}]},
]}
for name, payload in [("instructions", a), ("developer message", b), ("content parts", c)]:
    print(f"--- {name}\n{json.dumps(payload, indent=1)}")

In [ ]:
# (live) instructions vs developer message: roughly equivalent for a single request
for payload in (a, b):
    r = client.responses.create(**payload, reasoning={"effort": "low"}, max_output_tokens=600)
    print(r.status, "|", r.output_text[:150], "\n")

In [ ]:
# (live) instructions are NOT carried over by previous_response_id
RULE = "Always answer in French."
r1 = client.responses.create(model=MODEL, instructions=RULE, input="Name one cloud provider.",
                             reasoning={"effort": "low"}, max_output_tokens=400)
no_resend = client.responses.create(model=MODEL, previous_response_id=r1.id,
                                    input="Name another one.", reasoning={"effort": "low"}, max_output_tokens=400)
resend = client.responses.create(model=MODEL, instructions=RULE, previous_response_id=r1.id,
                                 input="Name another one.", reasoning={"effort": "low"}, max_output_tokens=400)
print("turn 1           :", r1.output_text)
print("turn 2 no resend :", no_resend.output_text)
print("turn 2 resend    :", resend.output_text)

# These responses were stored (store defaults to True). Delete them if you don't need them.
for rid in (r1.id, no_resend.id, resend.id):
    client.responses.delete(rid)

> **Observe:** without the resent rule, turn 2 may or may not stay in French; the model only sees the earlier *answer*, not the instruction. Don't rely on luck: resend `instructions` every turn.
>
> **Try it:** move the rule into a `{"role": "developer", ...}` item in turn 1's `input` and chain without resending. It now travels with the chain because it is an input item.

## 1.3 Status and incomplete_details: why generation stopped

There is no `stop_reason`. Read `status` (`completed`, `incomplete`, `failed`, `cancelled`, `queued`, `in_progress`), then `incomplete_details.reason` (`max_output_tokens`, `content_filter`, …), then the output item types (`function_call`, `refusal` parts). [Theory](README.md#13-status-and-incomplete_details-why-generation-stopped)

In [ ]:
# (live) Force an incomplete response with a tiny budget
r = client.responses.create(
    model=MODEL,
    input="Write a detailed 800-word essay on the CAP theorem with examples.",
    reasoning={"effort": "medium"},
    max_output_tokens=32,           # far too small: reasoning alone can use it all
)
show(r)
if r.status == "incomplete" and r.incomplete_details.reason == "max_output_tokens":
    print("Partial output" if r.output_text else "Budget spent during reasoning: nothing visible, still billed")

In [ ]:
# (offline) A reusable handler, tested on every case with mock responses
def handle(resp) -> str | None:
    match resp.status:
        case "completed":
            if any(item.type == "function_call" for item in resp.output):
                raise NotImplementedError("Run the tools and loop (Chapter 2)")
            refusals = [p.refusal for item in resp.output if item.type == "message"
                        for p in item.content if p.type == "refusal"]
            if refusals:
                print("  refused:", refusals[0])
                return None
            return resp.output_text
        case "incomplete":
            reason = resp.incomplete_details.reason if resp.incomplete_details else None
            if reason == "max_output_tokens":
                if resp.output_text:
                    raise RuntimeError("Truncated answer: raise max_output_tokens or lower effort")
                raise RuntimeError("Budget spent on reasoning: no visible output")
            raise RuntimeError(f"Incomplete: {reason}")
        case "failed":
            raise RuntimeError(f"Failed: {resp.error.code if resp.error else 'unknown'}")
        case "queued" | "in_progress":
            raise RuntimeError("Still running: poll responses.retrieve(resp.id)")
        case "cancelled":
            return None
        case other:
            raise ValueError(f"Unknown status: {other}")


cases = {
    "answer": mock_response(text="Paris."),
    "tool call": mock_response(text="Let me check.", function_call="get_order"),
    "refusal": mock_response(refusal="I can't help with that."),
    "truncated": mock_response(status="incomplete", reason="max_output_tokens", text="The CAP theorem states"),
    "reasoning ate budget": mock_response(status="incomplete", reason="max_output_tokens"),
    "content filter": mock_response(status="incomplete", reason="content_filter", text="partial"),
    "failed": mock_response(status="failed", error={"code": "server_error", "message": "boom"}, reasoning=False),
    "queued": mock_response(status="queued", reasoning=False),
}
for name, resp in cases.items():
    try:
        print(f"{name:22} -> {handle(resp)!r}")
    except Exception as e:  # noqa: BLE001 - we want to see every branch
        print(f"{name:22} -> {type(e).__name__}: {e}")

> **Observe:** the `tool call` case has text **and** a `function_call` item. A loop that stops on "has text" would end too early.
>
> **Try it:** add a branch for `incomplete_details.reason == "steered"` that returns the partial text and logs that a successor response will follow.

## 1.4 Conversation state: stateless, chained or durable

Three patterns: manual history with `store=False`; `previous_response_id` (responses stored 30 days by default); the Conversations API (stored until deleted; `conversations.delete()` does not delete the items, so delete those first). `previous_response_id` and `conversation` cannot be combined, and every pattern bills the full context as input. [Theory](README.md#14-conversation-state-stateless-chained-or-durable)

In [ ]:
# (live) A) Stateless: your app owns the transcript
history = [{"role": "user", "content": "I'm vegetarian and I love hiking."}]
r1 = client.responses.create(model=MODEL, input=history, store=False,
                             reasoning={"effort": "low"}, max_output_tokens=600)
history += r1.output          # keep every output item, including encrypted reasoning
history.append({"role": "user", "content": "Suggest one weekend trip near Almaty in 2 sentences."})
r2 = client.responses.create(model=MODEL, input=history, store=False,
                             reasoning={"effort": "low"}, max_output_tokens=600)
print("items replayed:", len(history), "| input tokens turn1/turn2:", r1.usage.input_tokens, r2.usage.input_tokens)
print(r2.output_text)

In [ ]:
# (live) B) Chained with previous_response_id (resend instructions every turn)
RULES = "You are a friendly travel assistant. Keep answers under 60 words."
c1 = client.responses.create(model=MODEL, instructions=RULES, input="I'm vegetarian and I love hiking.",
                             reasoning={"effort": "low"}, max_output_tokens=600)
c2 = client.responses.create(model=MODEL, instructions=RULES, previous_response_id=c1.id,
                             input="Suggest one weekend trip near Almaty.",
                             reasoning={"effort": "low"}, max_output_tokens=600)
print("previous_response_id:", c2.previous_response_id == c1.id,
      "| input tokens turn1/turn2:", c1.usage.input_tokens, c2.usage.input_tokens)
print(c2.output_text)
for rid in (c1.id, c2.id):
    client.responses.delete(rid)   # stored responses otherwise stay ~30 days

In [ ]:
# (live) C) Durable Conversations API
conv = client.conversations.create(metadata={"demo": "chapter-01"})
try:
    for text in ["I'm vegetarian and I love hiking.", "Suggest one weekend trip near Almaty."]:
        r = client.responses.create(model=MODEL, instructions=RULES, conversation=conv.id, input=text,
                                    reasoning={"effort": "low"}, max_output_tokens=600)
        print(f"[in={r.usage.input_tokens}] {r.output_text[:200]}")
    items = list(client.conversations.items.list(conv.id, order="asc"))   # iterating auto-paginates
    print("items stored in the conversation:", [item.type for item in items])
finally:
    # No 30-day TTL, and deleting the conversation does NOT delete its items: delete them first.
    for item_id in [item.id for item in client.conversations.items.list(conv.id)]:
        client.conversations.items.delete(item_id, conversation_id=conv.id)
    client.conversations.delete(conv.id)

In [ ]:
# (offline) What can't be combined
def check_state_kwargs(kwargs: dict) -> list[str]:
    problems = []
    if kwargs.get("previous_response_id") and kwargs.get("conversation"):
        problems.append("previous_response_id and conversation cannot be used together")
    if kwargs.get("previous_response_id") and kwargs.get("store") is False:
        problems.append("store=False on a chained turn: this response is not persisted, so the next HTTP turn "
                        "cannot chain from it (only WebSocket mode keeps an in-memory copy); replay history instead")
    if (kwargs.get("previous_response_id") or kwargs.get("conversation")) and "instructions" not in kwargs:
        problems.append("instructions are not carried over: resend them")
    return problems


print(check_state_kwargs({"previous_response_id": "resp_1", "conversation": "conv_1", "input": "hi"}))
print(check_state_kwargs({"previous_response_id": "resp_1", "store": False, "instructions": "x", "input": "hi"}))
print(check_state_kwargs({"conversation": "conv_1", "instructions": "x", "input": "hi"}))

> **Observe:** input tokens grow on turn 2 in **all three** patterns. Server-side state saves you code, not input billing.
>
> **Try it:** in pattern A, replay only the assistant *text* instead of `r1.output` and compare `usage` and the answer.

## 1.5 Streaming, background mode and webhooks

`stream=True` gives typed server-sent events (`response.output_text.delta`, `response.completed`, …). `background=True` starts a job you poll with `responses.retrieve` and can `cancel`. Webhooks (`response.completed`) replace polling in production. [Theory](README.md#15-streaming-background-mode-and-webhooks)

In [ ]:
# (live) Raw streaming: branch on event.type
stream = client.responses.create(model=MODEL, input="Explain idempotency in 3 short bullets.",
                                 reasoning={"effort": "low"}, max_output_tokens=800, stream=True)
seen = set()
final = None
for event in stream:
    seen.add(event.type)
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type in ("response.completed", "response.incomplete", "response.failed"):
        final = event.response
print("\n\nevent types seen:", sorted(seen))
if final:
    print("final status:", final.status, "| total tokens:", final.usage.total_tokens)

In [ ]:
# (live) The SDK helper accumulates the final Response for you
with client.responses.stream(model=MODEL, input="One sentence: what is a webhook?",
                             reasoning={"effort": "low"}, max_output_tokens=400) as s:
    for event in s:
        if event.type == "response.output_text.delta":
            print(event.delta, end="")
    final = s.get_final_response()
print("\n", final.status, final.usage.output_tokens)

In [ ]:
# (live) Background mode: start, poll with a deadline, cancel if it takes too long
job = client.responses.create(model=MODEL, input="List 5 risks of a database migration, one line each.",
                              reasoning={"effort": "low"}, max_output_tokens=1000, background=True)
print("started:", job.id, job.status)
deadline = time.time() + 60
while job.status in {"queued", "in_progress"} and time.time() < deadline:
    time.sleep(2)
    job = client.responses.retrieve(job.id)
    print("  status:", job.status)
if job.status in {"queued", "in_progress"}:
    job = client.responses.cancel(job.id)   # idempotent
print("final:", job.status)
print(job.output_text[:500])

In [ ]:
# (offline) Accumulate a stream from fake events: the logic your UI layer needs
fake_events = [
    SimpleNamespace(type="response.created", sequence_number=0),
    SimpleNamespace(type="response.output_text.delta", delta="Idempotent ", sequence_number=1),
    SimpleNamespace(type="response.output_text.delta", delta="means safe ", sequence_number=2),
    SimpleNamespace(type="response.output_text.delta", delta="to retry.", sequence_number=3),
    SimpleNamespace(type="response.completed", sequence_number=4,
                    response=SimpleNamespace(status="completed", usage=SimpleNamespace(total_tokens=57))),
]


def collect(events):
    text, status, cursor = [], None, None
    for ev in events:
        cursor = ev.sequence_number        # keep this to resume a background stream
        if ev.type == "response.output_text.delta":
            text.append(ev.delta)
        elif ev.type in ("response.completed", "response.incomplete", "response.failed"):
            status = ev.response.status
    return "".join(text), status, cursor


print(collect(fake_events))

> **Observe:** the stream ends with a `response.completed` (or `.incomplete` / `.failed`) event that carries the full `Response`, including `usage`. Read the status there, not from the last delta.
>
> **Try it:** create the background job with both `background=True` and `stream=True`, print `event.sequence_number`, and think about where you would persist it to resume after a disconnect.

## 1.6 Context window, tokens and cost

GPT-6 models: 1,050,000-token window, 922,000 max input, 128,000 max output. Reasoning tokens fill the window and bill as output. Count before sending with `responses.input_tokens.count`, measure after with `usage`. Prompts above 272K input tokens cost 2x input and 1.5x output. [Theory](README.md#16-context-window-tokens-and-cost)

In [ ]:
# (live) Count before, measure after
payload = dict(
    model=MODEL,
    instructions="You are a concise assistant.",
    input=[{"role": "user", "content": "Summarize the benefits of message queues in 3 bullets."}],
)
before = client.responses.input_tokens.count(**payload)   # context fields only: max_output_tokens, store, stream raise TypeError here
after = client.responses.create(**payload, reasoning={"effort": "low"}, max_output_tokens=600)
print("counted input tokens:", before.input_tokens, "| billed input tokens:", after.usage.input_tokens)
print("output tokens:", after.usage.output_tokens,
      "of which reasoning:", after.usage.output_tokens_details.reasoning_tokens)
print(f"cost ≈ ${cost_usd(after.usage, after.model):.6f}")

In [ ]:
# (offline) Cost of the same usage on each model, and the long-context surcharge
def usage_ns(inp, out, cached=0, writes=0, reasoning=0):
    return SimpleNamespace(
        input_tokens=inp, output_tokens=out,
        input_tokens_details=SimpleNamespace(cached_tokens=cached, cache_write_tokens=writes),
        output_tokens_details=SimpleNamespace(reasoning_tokens=reasoning))


u_small = usage_ns(10_000, 1_000)
u_long = usage_ns(300_000, 1_000)
for m in PRICES:
    print(f"{m:12}  10K-in: ${cost_usd(u_small, m):.4f}   300K-in: ${cost_usd(u_long, m):.4f}")

In [ ]:
# (offline) Input tokens re-billed per turn: the same for manual history and previous_response_id
TURN_TOKENS = 150        # user + assistant tokens added per turn
SYSTEM_TOKENS = 1_200    # instructions + tool definitions (stable prefix, cache-friendly)
billed = []
for turn in range(1, 21):
    billed.append(SYSTEM_TOKENS + TURN_TOKENS * (turn - 1) + 60)  # prior turns + new user message
print("input tokens on turn 1, 10, 20:", billed[0], billed[9], billed[19])
print("total input billed over 20 turns:", sum(billed))
cached_share = SYSTEM_TOKENS * 20 / sum(billed)
print(f"share of that input that is the stable prefix (cache candidate): {cached_share:.0%}")

> **Observe:** the counter and `usage.input_tokens` agree closely. Long chats grow linearly per turn and the total grows quadratically; caching makes the stable prefix cheaper, compaction makes the context smaller.
>
> **Try it:** add a tool definition to `payload` and count again. Tools cost input tokens on every request.

## 1.7 Model selection and reasoning effort

Astra ($10/$50), 6.1 Sol ($2/$10), Luna ($0.10/$0.50). `reasoning.effort` values are model-specific: Astra and 6.1 Sol don't accept `none`; Luna does. Choose from evals, comparing cost per successful task. [Theory](README.md#17-model-selection-and-reasoning-effort)

In [ ]:
# (offline) Validate effort per model before you send a request (unsupported values return HTTP 400)
SUPPORTED_EFFORT = {
    "gpt-6-astra": {"low", "medium", "high", "xhigh", "max"},
    "gpt-6.1-sol": {"low", "medium", "high", "xhigh", "max"},
    "gpt-6-sol": {"none", "low", "medium", "high", "xhigh", "max"},
    "gpt-6-luna": {"none", "low", "medium", "high", "xhigh", "max"},
}


def reasoning_param(model: str, effort: str) -> dict:
    allowed = SUPPORTED_EFFORT.get(model)
    if allowed is not None and effort not in allowed:
        raise ValueError(f"{model} does not support effort={effort!r}; allowed: {sorted(allowed)}")
    return {"effort": effort}


for m, e in [("gpt-6-luna", "none"), ("gpt-6-astra", "none"), ("gpt-6.1-sol", "minimal"), ("gpt-6-astra", "low")]:
    try:
        print(m, e, "->", reasoning_param(m, e))
    except ValueError as err:
        print(m, e, "->", err)

In [ ]:
# (live) Effort sweep on the same small task: tokens, latency, cost
TASK = "A train leaves at 09:40 and the trip takes 2 h 35 min. When does it arrive? Answer with the time only."
rows = []
for effort in ["none", "low", "medium"]:
    try:
        rp = reasoning_param(MODEL, effort)
    except ValueError as err:
        print("skip:", err)
        continue
    t0 = time.perf_counter()
    r = client.responses.create(model=MODEL, input=TASK, reasoning=rp, max_output_tokens=2000)
    rows.append((effort, r.output_text.strip(), r.usage.output_tokens_details.reasoning_tokens,
                 round(time.perf_counter() - t0, 2), cost_usd(r.usage, r.model)))
for row in rows:
    print(f"effort={row[0]:7} answer={row[1]!r:10} reasoning_tokens={row[2]:5} latency={row[3]}s cost=${row[4]:.6f}")

In [ ]:
# (live) Lifecycle check: the Models API exposes a shutdown date (context limits live on the model page)
m = client.models.retrieve(MODEL)
print(m.id, "| owned_by:", m.owned_by, "| shutdown_date:", m.shutdown_date)

> **Observe:** for an easy task, higher effort mostly adds reasoning tokens and latency, not accuracy. That is why you pick effort from an eval, not by habit.
>
> **Try it:** set `OPENAI_MODEL=gpt-6.1-sol` in `.env`, rerun the sweep, and see `none` get skipped by the validator instead of failing with a 400.

### Fast front, flagship behind (offline)

OpenAI's Building Agents track: a fast model chats with the user and **delegates demanding tasks** to the flagship. Below, Luna at `none` answers small talk itself and calls a `delegate` function when a request needs real planning; only then does the router call Astra. Fake `create` functions stand in for the API, so you can see the routing and the cost difference without spending anything.

In [ ]:
# (offline) A two-tier router: Luna at the front, Astra only when Luna asks for help
DELEGATE_TOOL = {
    "type": "function", "name": "delegate",
    "description": "Hand a demanding planning or analysis task to a stronger model.",
    "parameters": {"type": "object", "properties": {"task": {"type": "string"}},
                   "required": ["task"], "additionalProperties": False},
    "strict": True,
}


def route(user_text: str, create) -> tuple[str, list]:
    """Return (answer, [(model, usage), ...]). `create` has the signature of client.responses.create."""
    billed = []
    fast = create(model="gpt-6-luna", reasoning={"effort": "none"}, tools=[DELEGATE_TOOL],
                  instructions="You are a friendly travel assistant. For multi-step planning, call delegate.",
                  input=user_text, store=False)
    billed.append(("gpt-6-luna", fast.usage))
    calls = [item for item in fast.output if item.type == "function_call" and item.name == "delegate"]
    if not calls:
        return fast.output_text, billed
    task = json.loads(calls[0].arguments)["task"]
    deep = create(model="gpt-6-astra", reasoning={"effort": "high"}, input=task, store=False)
    billed.append(("gpt-6-astra", deep.usage))
    return deep.output_text, billed


def fake_create(**kw):
    """Pretend Luna delegates anything that mentions an itinerary."""
    if kw["model"] == "gpt-6-luna" and "itinerary" in str(kw["input"]).lower():
        r = mock_response(function_call="delegate", reasoning=False, usage=(300, 30, 0), model="gpt-6-luna")
        r.output[-1].arguments = json.dumps({"task": kw["input"]})
        return r
    if kw["model"] == "gpt-6-luna":
        return mock_response(text="Happy to help! Where are you headed?", reasoning=False,
                             usage=(300, 20, 0), model="gpt-6-luna")
    return mock_response(text="Day 1: Almaty ... Day 5: Bishkek ...", usage=(400, 3000, 2200), model="gpt-6-astra")


for msg in ["Hi there!", "Plan a 5-day itinerary: Almaty, Issyk-Kul, Bishkek."]:
    answer, billed = route(msg, fake_create)
    cost = sum(cost_usd(u, m) for m, u in billed)
    print(f"{msg!r}\n  models={[m for m, _ in billed]} cost=${cost:.6f}\n  -> {answer}")

> **Observe:** small talk costs a fraction of a cent on Luna alone; only the planning request pays for Astra and its reasoning tokens. In a real product the deep call would usually run with `background=True` (1.5), and the delegation decision itself belongs in your eval set.

## 1.8 Chat Completions: the legacy contrast

Chat Completions is still supported but not recommended for new work. It returns `choices[0].message.content` and `finish_reason`; Responses returns typed `output` items and `status`. On GPT-6 Astra and 6.1 Sol, tool calling needs Responses. [Theory](README.md#18-chat-completions-the-legacy-contrast)

In [ ]:
# (live) Same prompt, both APIs
messages = [{"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
            {"role": "user", "content": "What is a load balancer?"}]

completion = client.chat.completions.create(model=MODEL, messages=messages,
                                            reasoning_effort="low", max_completion_tokens=600)
print("Chat Completions:", completion.choices[0].finish_reason, "|", completion.choices[0].message.content)

response = client.responses.create(model=MODEL, input=messages,   # text-only message lists are accepted as-is
                                   reasoning={"effort": "low"}, max_output_tokens=600)
print("Responses       :", response.status, "|", response.output_text)

In [ ]:
# (offline) Migrate a Chat Completions message list to Responses kwargs
def chat_to_responses(messages: list[dict]) -> dict:
    """Move leading system/developer messages into `instructions`; keep the rest as input items."""
    head = []
    rest = list(messages)
    while rest and rest[0]["role"] in ("system", "developer"):
        head.append(rest.pop(0)["content"])
    kwargs = {"input": rest}
    if head:
        kwargs["instructions"] = "\n\n".join(head)
    return kwargs


FINISH_TO_STATUS = {   # rough mental mapping, not an API feature
    "stop": "status=completed, no function_call items",
    "tool_calls": "status=completed, function_call items present",
    "length": "status=incomplete, reason=max_output_tokens",
    "content_filter": "status=incomplete, reason=content_filter (or a refusal part)",
}
print(json.dumps(chat_to_responses(messages), indent=1))
for k, v in FINISH_TO_STATUS.items():
    print(f"{k:15} -> {v}")

> **Observe:** the Responses call returned a `status` instead of a `finish_reason`, and no `choices` list (there is no `n` parameter).

## 1.9 Playground to code, sampling and non-determinism

Prototype in the Playground, then move the settled prompt and parameters into versioned code (saved prompt objects shut down on 2026-11-30). Output is sampled, so identical requests can differ; on GPT-6 with reasoning on you can't even set `temperature`. Measure pass rates over several runs. [Theory](README.md#19-playground-to-code-sampling-and-non-determinism)

In [ ]:
# (offline) Turn the settings you settled on in the Playground into a request you can check into the repo
GPT6 = ("gpt-6-astra", "gpt-6.1-sol", "gpt-6-sol", "gpt-6-luna")
SAMPLING_KEYS = ("temperature", "top_p", "top_logprobs")


def playground_to_request(settings: dict) -> tuple[dict, list[str]]:
    """Return (kwargs for responses.create, warnings). Drops what the Responses API or the model won't take."""
    s = dict(settings)
    warnings = []
    if s.pop("seed", None) is not None:
        warnings.append("dropped seed: responses.create has no seed parameter")
    if "prompt_id" in s:
        s.pop("prompt_id")
        warnings.append("dropped prompt_id: prompt objects shut down 2026-11-30; inline the prompt as instructions")
    effort = s.pop("effort", None)
    kwargs = {"model": s.pop("model"), "instructions": s.pop("instructions"), "input": s.pop("input", "")}
    if effort:
        kwargs["reasoning"] = reasoning_param(kwargs["model"], effort)   # validated per model (1.7)
    if "verbosity" in s:
        kwargs["text"] = {"verbosity": s.pop("verbosity")}
    reasoning_on = effort != "none"   # GPT-6 models reason by default when effort is omitted
    for key in SAMPLING_KEYS:
        if key in s:
            value = s.pop(key)
            if kwargs["model"].startswith(GPT6) and reasoning_on:
                warnings.append(f"dropped {key}={value}: OpenAI says to remove it on GPT-6 unless reasoning effort is 'none'")
            else:
                kwargs[key] = value
    if s:
        warnings.append(f"unmapped settings, review by hand: {sorted(s)}")
    return kwargs, warnings


from_playground = {"model": "gpt-6.1-sol", "instructions": "Classify the ticket: billing, technical or other.",
                   "effort": "low", "verbosity": "low", "temperature": 0, "seed": 42, "prompt_id": "pmpt_123"}
kwargs, warnings = playground_to_request(from_playground)
print(json.dumps(kwargs, indent=1))
print(*warnings, sep="\n")
print(playground_to_request({"model": "gpt-6-luna", "instructions": "x", "effort": "none", "temperature": 0.2})[0])

In [ ]:
# (offline) What temperature does to one next-token pick (toy logits, no API)
import math
import random
from collections import Counter

LOGITS = {"billing": 2.0, "technical": 1.6, "other": 0.3}


def sample(logits: dict, temperature: float, rng: random.Random) -> str:
    if temperature == 0:
        return max(logits, key=logits.get)               # greedy: always the top token
    weights = {t: math.exp(v / temperature) for t, v in logits.items()}
    r, acc = rng.random() * sum(weights.values()), 0.0
    for token, w in weights.items():
        acc += w
        if r <= acc:
            return token
    return token


rng = random.Random(7)
for t in (0, 0.2, 1.0, 2.0):
    counts = Counter(sample(LOGITS, t, rng) for _ in range(1000))
    print(f"temperature={t:<4} ->", dict(counts))

In [ ]:
# (offline) Why evals sample several runs: a pass rate, not one lucky run
def pass_rate(run_once, examples: list[dict], runs: int = 5) -> dict:
    """run_once(example) -> label. Returns the overall rate and the per-example rates."""
    per_example = {}
    for ex in examples:
        hits = sum(run_once(ex) == ex["expected"] for _ in range(runs))
        per_example[ex["id"]] = hits / runs
    overall = sum(per_example.values()) / len(per_example)
    return {"overall": round(overall, 2), "per_example": per_example}


EXAMPLES = [{"id": "t1", "text": "Charged twice in September", "expected": "billing"},
            {"id": "t2", "text": "App crashes on login", "expected": "technical"},
            {"id": "t3", "text": "Refund for a failed upgrade?", "expected": "billing"}]   # ambiguous on purpose
rng = random.Random(1)


def flaky_model(ex):   # stands in for a real call: right most of the time, unsure on the ambiguous ticket
    if ex["id"] == "t3":
        return rng.choice(["billing", "technical"])
    return ex["expected"] if rng.random() < 0.95 else "other"


print("1 run :", pass_rate(flaky_model, EXAMPLES, runs=1))
print("5 runs:", pass_rate(flaky_model, EXAMPLES, runs=5))

In [ ]:
# (live) Same request five times: count distinct answers (about 5 tiny Luna calls)
req, _ = playground_to_request({"model": MODEL, "instructions": "Reply with one word: billing, technical or other.",
                                "effort": "low", "input": "I was charged after my upgrade failed."})
answers = []
for _ in range(5):
    r = client.responses.create(**req, max_output_tokens=500, store=False)
    answers.append(r.output_text.strip().lower())
print(Counter(answers))

> **Observe:** at `temperature=0` the toy sampler always picks the top token; at 1.0 and 2.0 the tail shows up. On GPT-6 with reasoning on you don't get that knob at all, so the pass rate over several runs is your measure. One run said little about the ambiguous ticket `t3`.
>
> **Try it:** add an example to `EXAMPLES` that the flaky model always gets right, and see why a clear-cut eval set can hide instability. Then move `from_playground` into a `prompts/` module of your own.

## Mini-project: a multi-turn chat helper

Combine the chapter in one small class, `ChatSession`:

- **state** via `mode="chain"` (`previous_response_id`), `"conversation"` (Conversations API) or `"stateless"` (manual history, `store=False`)
- **instructions resent** on every request
- **incomplete handling:** on `max_output_tokens`, retry once with double the budget (chain/stateless); never commit a failed turn to state
- **usage and cost report** across every billed response, including the incomplete ones

It is tested offline against a fake client first, then run live.

In [ ]:
@dataclass
class TurnResult:
    text: str
    status: str
    reason: str | None
    attempts: int


@dataclass
class ChatSession:
    client: object
    model: str
    instructions: str
    mode: str = "chain"                 # "chain" | "conversation" | "stateless"
    effort: str = "low"
    max_output_tokens: int = 800
    retries_on_truncation: int = 1
    history: list = field(default_factory=list)
    last_response_id: str | None = None
    conversation_id: str | None = None
    usage_log: list = field(default_factory=list)

    def __post_init__(self):
        if self.mode not in {"chain", "conversation", "stateless"}:
            raise ValueError(f"unknown mode {self.mode!r}")

    def _request(self, user_text: str, budget: int) -> dict:
        kwargs = dict(model=self.model, instructions=self.instructions,
                      reasoning={"effort": self.effort}, max_output_tokens=budget)
        user_msg = {"role": "user", "content": user_text}
        if self.mode == "stateless":
            kwargs.update(store=False, input=self.history + [user_msg])
        elif self.mode == "chain":
            kwargs["input"] = [user_msg]
            if self.last_response_id:
                kwargs["previous_response_id"] = self.last_response_id
        else:
            if self.conversation_id is None:
                self.conversation_id = self.client.conversations.create().id
            kwargs.update(conversation=self.conversation_id, input=[user_msg])
        return kwargs

    def send(self, user_text: str) -> TurnResult:
        budget = self.max_output_tokens
        for attempt in range(1, self.retries_on_truncation + 2):
            resp = self.client.responses.create(**self._request(user_text, budget))
            self.usage_log.append((resp.model, resp.usage))   # incomplete responses are billed too
            reason = resp.incomplete_details.reason if resp.incomplete_details else None
            if resp.status == "completed":
                self._commit(user_text, resp)
                refusals = [p.refusal for item in resp.output if item.type == "message"
                            for p in item.content if p.type == "refusal"]
                return TurnResult(refusals[0] if refusals else resp.output_text, "completed", None, attempt)
            truncated = resp.status == "incomplete" and reason == "max_output_tokens"
            # Conversation mode: the API reference says items are added to the conversation "after this
            # response completes" and is silent about incomplete responses. So we don't blindly resend
            # the same user message; check conversations.items.list(...) before retrying by hand.
            if truncated and self.mode != "conversation" and attempt <= self.retries_on_truncation:
                budget *= 2
                continue
            return TurnResult(resp.output_text, resp.status, reason, attempt)
        raise AssertionError("unreachable")

    def _commit(self, user_text: str, resp) -> None:
        if self.mode == "stateless":
            self.history.append({"role": "user", "content": user_text})
            self.history += resp.output          # keep every item, incl. encrypted reasoning
        elif self.mode == "chain":
            self.last_response_id = resp.id      # only completed turns become the chain head

    def report(self) -> dict:
        totals = {"responses": len(self.usage_log), "input": 0, "cached": 0, "output": 0, "reasoning": 0, "cost_usd": 0.0}
        for model, u in self.usage_log:
            if u is None:
                continue
            totals["input"] += u.input_tokens
            totals["cached"] += u.input_tokens_details.cached_tokens
            totals["output"] += u.output_tokens
            totals["reasoning"] += u.output_tokens_details.reasoning_tokens
            totals["cost_usd"] += cost_usd(u, model)
        totals["cost_usd"] = round(totals["cost_usd"], 6)
        return totals

    def close(self) -> None:
        if self.conversation_id:
            conv = self.conversation_id
            # conversations.delete() leaves the items behind, so remove the transcript first
            for item_id in [item.id for item in self.client.conversations.items.list(conv)]:
                self.client.conversations.items.delete(item_id, conversation_id=conv)
            self.client.conversations.delete(conv)
            self.conversation_id = None

In [ ]:
# (offline) Test the helper against a scripted fake client
class FakeResponses:
    def __init__(self, script):
        self.script, self.calls = list(script), []

    def create(self, **kwargs):
        self.calls.append(kwargs)
        return self.script.pop(0)


class FakeItems:
    def __init__(self, store):
        self.store, self.deleted = store, []

    def list(self, conversation_id, **kwargs):
        return [SimpleNamespace(id=i) for i in self.store.get(conversation_id, [])]

    def delete(self, item_id, *, conversation_id):
        self.store[conversation_id].remove(item_id)
        self.deleted.append(item_id)


class FakeConversations:
    def __init__(self):
        self.created, self.deleted, self.store = 0, [], {}
        self.items = FakeItems(self.store)

    def create(self, **kwargs):
        self.created += 1
        self.store[f"conv_{self.created}"] = []
        return SimpleNamespace(id=f"conv_{self.created}")

    def delete(self, conversation_id):   # like the real API: the items are NOT deleted
        self.deleted.append(conversation_id)


class FakeResponsesWithConversation(FakeResponses):
    def __init__(self, script, conversations):
        super().__init__(script)
        self.conversations = conversations

    def create(self, **kwargs):
        resp = super().create(**kwargs)
        if kwargs.get("conversation"):   # the API appends the turn's input and output items
            self.conversations.store[kwargs["conversation"]] += [f"msg_in_{len(self.calls)}", f"msg_out_{resp.id}"]
        return resp


class FakeClient:
    def __init__(self, script):
        self.conversations = FakeConversations()
        self.responses = FakeResponsesWithConversation(script, self.conversations)


fake = FakeClient([
    mock_response(rid="resp_1", text="Noted: vegetarian, likes hiking."),
    mock_response(rid="resp_2", status="incomplete", reason="max_output_tokens", usage=(200, 800, 800)),
    mock_response(rid="resp_3", text="Try the Big Almaty Lake trail."),
])
s = ChatSession(fake, model="gpt-6-luna", instructions="Be brief.", mode="chain", max_output_tokens=800)
t1, t2 = s.send("I'm vegetarian and like hiking."), s.send("Suggest a trip.")
calls = fake.responses.calls
assert t1.status == t2.status == "completed" and t2.attempts == 2
assert "previous_response_id" not in calls[0]
assert calls[1]["previous_response_id"] == calls[2]["previous_response_id"] == "resp_1"   # retry from last good turn
assert calls[2]["max_output_tokens"] == 1600                                              # budget doubled
assert all(c["instructions"] == "Be brief." for c in calls)                               # resent every time
assert s.last_response_id == "resp_3" and s.report()["responses"] == 3                   # 3 billed responses
print("chain mode OK:", s.report())

fake = FakeClient([mock_response(rid="r1", text="ok"), mock_response(rid="r2", text="ok again")])
s = ChatSession(fake, model="gpt-6-luna", instructions="Be brief.", mode="conversation")
s.send("hi"), s.send("again")
assert {c["conversation"] for c in fake.responses.calls} == {"conv_1"}
assert all("previous_response_id" not in c for c in fake.responses.calls)
s.close()
assert fake.conversations.deleted == ["conv_1"]
assert fake.conversations.store["conv_1"] == [] and len(fake.conversations.items.deleted) == 4   # items first
print("conversation mode OK (items deleted, then the conversation)")

fake = FakeClient([mock_response(rid="r1", text="ok"), mock_response(rid="r2", text="ok again")])
s = ChatSession(fake, model="gpt-6-luna", instructions="Be brief.", mode="stateless")
s.send("hi"), s.send("again")
second = fake.responses.calls[1]
assert second["store"] is False and second["input"][0] == {"role": "user", "content": "hi"}
assert len(second["input"]) == 1 + 2 + 1   # user, reasoning + message items, new user message
print("stateless mode OK")

In [ ]:
# (live) Run it for real
session = ChatSession(client, model=MODEL, mode="chain",
                      instructions="You are a friendly travel assistant. Keep answers under 60 words.")
try:
    for msg in ["I'm vegetarian and I love hiking.",
                "Suggest one weekend trip near Almaty.",
                "What should I eat there, given what I told you?"]:
        result = session.send(msg)
        print(f"> {msg}\n[{result.status}, attempts={result.attempts}] {result.text}\n")
    print("usage report:", session.report())
finally:
    session.close()

> **Try it:** run the live cell with `mode="conversation"` and with `mode="stateless"`. Compare `report()["input"]`: the state pattern changes where history lives, not how much input you pay for.

## Recap

- One endpoint: `responses.create`. Read `output` by item type; `output_text` is a convenience, not a contract.
- There is no `stop_reason`: check `status`, then `incomplete_details.reason`, then the item types.
- `max_output_tokens` includes reasoning. Too small means an empty, billed answer.
- State: manual history (`store=False`), `previous_response_id` (30-day storage), or Conversations (until deleted: delete the items, then the conversation). Resend `instructions` every turn; never combine chain and conversation.
- Streaming for UX, background mode for long jobs, webhooks instead of polling.
- Count tokens with `responses.input_tokens.count`; price with `usage`, including cached and reasoning tokens.
- Choose model and effort from evals; validate effort per model. A fast front model can delegate demanding work to the flagship.
- Prototype in the Playground, then move prompts into versioned code. Outputs vary between runs: measure pass rates, don't rely on `temperature=0`.

Next: [homework](02_homework.ipynb)